In [1]:
import requests

In [2]:
from bs4 import BeautifulSoup as bs

In [3]:
base_url="https://books.toscrape.com/catalogue/category/books_1/page-1.html"
response=requests.get(base_url)

print(response.text)



<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    Books | 
     Books to Scrape - Sandbox

</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:30" />
        <meta name="description" content="
    
" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->

        
            <link rel="shortcut icon" href="../../../stat

In [4]:
soup=bs(response.text,"html.parser")
# soup

print(soup.prettify())

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js" lang="en-us">
 <!--<![endif]-->
 <head>
  <title>
   Books | 
     Books to Scrape - Sandbox
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="24th Jun 2016 09:30" name="created"/>
  <meta content="
    
" name="description"/>
  <meta content="width=device-width" name="viewport"/>
  <meta content="NOARCHIVE,NOCACHE" name="robots"/>
  <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
  <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
  <link href="../../../static/oscar/favicon.ico" rel="shortcut icon"/>
  <link href="../../../static/oscar/css/styles.css" rel

In [5]:
raw_books=[]
articles=soup.find_all("article", class_="product_pod")
# articles
len(articles)

20

# name of book

In [6]:
count=0
for book in articles:
    name=book.h3.a
    # count+=1
    # print(count)
    print(name)


    


<a href="../../a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a>
<a href="../../tipping-the-velvet_999/index.html" title="Tipping the Velvet">Tipping the Velvet</a>
<a href="../../soumission_998/index.html" title="Soumission">Soumission</a>
<a href="../../sharp-objects_997/index.html" title="Sharp Objects">Sharp Objects</a>
<a href="../../sapiens-a-brief-history-of-humankind_996/index.html" title="Sapiens: A Brief History of Humankind">Sapiens: A Brief History ...</a>
<a href="../../the-requiem-red_995/index.html" title="The Requiem Red">The Requiem Red</a>
<a href="../../the-dirty-little-secrets-of-getting-your-dream-job_994/index.html" title="The Dirty Little Secrets of Getting Your Dream Job">The Dirty Little Secrets ...</a>
<a href="../../the-coming-woman-a-novel-based-on-the-life-of-the-infamous-feminist-victoria-woodhull_993/index.html" title="The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull">The Comin

# all code for practicing

In [7]:
from urllib.parse import urljoin

In [ ]:
current_page=1
raw_books=[]
url=base_url
while url and current_page < 4:
    response=requests.get(url)
    soup=bs(response.text,"html.parser")
    articles=soup.find_all("article", class_="product_pod")
    len(articles)

    for book in articles:
        name=book.h3.a["title"]
        # price=book.find("p", class_="price_color")
        price=book.find("p", class_="price_color").text
        rating=book.find("p",class_="star-rating")["class"][1]

        img_src=book.find("img")["src"]
        img_url=urljoin(base_url,img_src)
        img_content=requests.get(img_url).content
        img_name=name.replace(":","")+".png"
        img_path=f"../raw_images/{img_name}"
        with open(img_path,"wb") as img_file:
            img_file.write(img_content)



        raw_books.append(
            {
                "name":name,
                "price":price,
                "rating":rating,
                "image_path":img_path
            }
        )

    next_btn=soup.find("li",class_="next")
    current_page+=1
    url=f"https://books.toscrape.com/catalogue/category/books_1/page-{current_page}.html" if next_btn else None


# save raw data

In [20]:
import pandas as pd
df=pd.DataFrame(raw_books)
df.to_csv("raw_data_of_books.csv")

# clean raw data

In [60]:
df=pd.read_csv("raw_data_of_books.csv")
df.head()

,Unnamed: 0,name,price,rating,image_path
0,0,A Light in the Attic,Â£51.77,Three,../raw_images/A Light in the Attic.png
1,1,Tipping the Velvet,Â£53.74,One,../raw_images/Tipping the Velvet.png
2,2,Soumission,Â£50.10,One,../raw_images/Soumission.png
3,3,Sharp Objects,Â£47.82,Four,../raw_images/Sharp Objects.png
4,4,Sapiens: A Brief History of Humankind,Â£54.23,Five,../raw_images/Sapiens A Brief History of Human...


In [61]:
def clean_data(df):
    df["price"] = df["price"].str.replace(r"[^\d.]", "", regex=True).astype(float)
    df["name"]=df["name"].astype("string")
    df["image_path"]=df["image_path"].astype("string")

    rating_map={
        "One":1,
        "Two":2,
        "Three":3,
        "Four":4,
        "Five":5
    }
    df["rating"]=df["rating"].map(rating_map)
    return df



In [62]:
clean_data(df).head()

,Unnamed: 0,name,price,rating,image_path
0,0,A Light in the Attic,51.77,3,../raw_images/A Light in the Attic.png
1,1,Tipping the Velvet,53.74,1,../raw_images/Tipping the Velvet.png
2,2,Soumission,50.10,1,../raw_images/Soumission.png
3,3,Sharp Objects,47.82,4,../raw_images/Sharp Objects.png
4,4,Sapiens: A Brief History of Humankind,54.23,5,../raw_images/Sapiens A Brief History of Human...


In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        60 non-null     string 
 1   price       60 non-null     float64
 2   rating      60 non-null     int64  
 3   image_path  60 non-null     string 
dtypes: float64(1), int64(1), string(2)
memory usage: 2.0 KB


In [75]:
df.drop("Unnamed: 0", axis=1, inplace=True)

In [77]:
df.to_csv("cleaned_data_of_books.csv" ,index=False)

# organizer

In [79]:
import os
import shutil

def organize_images_by_rating(df):
    base_folder = "images_by_rating"

    for i in range(1, 6):
        folder_path = os.path.join(base_folder, f"{i}_stars")
        os.makedirs(folder_path, exist_ok=True)

    for _, row in df.iterrows():
        image_path = row["image_path"]
        rating = row["rating"]

        if image_path and os.path.exists(image_path):
            file_name = os.path.basename(image_path)

            destination = os.path.join(
                base_folder,
                f"{rating}_stars",
                file_name
            )

            try:
                shutil.move(image_path, destination)
            except:
                print(f"Error moving {file_name}")

In [ ]:
organize_images_by_rating(df)


In [78]:
import os

base_folder = "ratings_data"

for i in range(1, 6):
    folder_path = os.path.join(base_folder, f"{i}_stars")
    os.makedirs(folder_path, exist_ok=True)

    df_rating = df[df["rating"] == i]
    
    file_path = os.path.join(folder_path, "data.csv")
    df_rating.to_csv(file_path, index=False)